# Count change of offered ECHConfigs over Test_Dates without ECH Deactivation

In [ ]:
import os
import pickle
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.pool import QueuePool

# Load environment variables from .env file
load_dotenv("../.env")  # Adjust the path if your .env file is in a different location


def fetch_and_save_data(query, filename_suffix, params=None, timeout=3600):
    """
    Fetches data from PostgreSQL using SQLAlchemy with connection pooling,
    saves it to a pickle file, and handles timeouts.
    Reads database credentials from environment variables.

    Args:
        query (str): The SQL query to execute.
        filename_suffix (str): A string to be added to the pickle filename.
        params (tuple, optional): Parameters to pass to the query. Defaults to None.
        timeout (int, optional): The timeout for the database connection in seconds. Defaults to 3600.
    """
    try:
        db_user = os.getenv("DB_USER")
        db_password = os.getenv("DB_PASSWORD")
        db_host = os.getenv("DB_HOST")
        db_name = os.getenv("DB_NAME")

        engine = create_engine(
            f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}/{db_name}",
            connect_args={"connect_timeout": timeout},
            poolclass=QueuePool,
            pool_size=5,
            max_overflow=10,
        )

        with engine.connect() as conn:
            df = pd.read_sql_query(query, conn, params=params)

        os.makedirs("./pickle", exist_ok=True)
        filename = f"./pickle/{datetime.now().strftime('%Y%m%d_%H%M%S')}_{filename_suffix}.pickle"

        with open(filename, "wb") as f:
            pickle.dump(df, f)

        print(f"Data saved to {filename}")

    except Exception as e:
        print(f"Error: {e}")


if __name__ == "__main__":
    query = """
SELECT hr1.test_date, COUNT(DISTINCT hr1.ech_config) AS ech_config_changes
FROM public."HTTPSRecords" hr1
LEFT JOIN public."HTTPSRecords" hr2 ON hr1.domain = hr2.domain AND hr2.test_date = (SELECT MAX(test_date) FROM public."HTTPSRecords" WHERE domain = hr1.domain AND test_date < hr1.test_date)
WHERE hr1.ech_config IS NOT NULL OR hr2.ech_config IS NOT NULL
GROUP BY hr1.test_date
ORDER BY hr1.test_date ASC;
    """
    filename_suffix = "ech_ciphe-suites_by_test_relation"
    fetch_and_save_data(query, filename_suffix)